# Extract 1 - Weather data

In [1]:
import boto3
import os
import pandas as pd
import json
import ast
import time
import gzip

In [2]:
s3_bucket = 'rwc-ml-datasets'
s3_path = 'raw/Flight_Delay_Prediction_Datasets/other_data/airports.json'

In [3]:
s3_client = boto3.client('s3')

In [4]:
airports = s3_client.get_object(
    Bucket=s3_bucket,
    Key=s3_path
)['Body'].read().decode('utf-8')

airport_list = json.loads(airports)

In [5]:
len(airport_list)

352

In [6]:
def fetch_batch(station_list, year="2025"):
    station_param = ",".join(station_list)
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
        f"station={station_param}"
        "&data=tmpf,dwpf,relh,sknt,gust,vsby,skyc1,skyc2,skyc3,wxcodes,p01i,alti"
        f"&year1={year}&month1=1&day1=1"
        f"&year2={int(year)+1}&month2=1&day2=1"
        "&tz=UTC&format=onlycomma&latlon=no&missing=M&trace=T&direct=no&report_type=3"
    )
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    return response.text

In [7]:
def write_to_datalake(raw_text: str, i):
    s3_key = f"raw/Flight_Delay_Prediction_Datasets/weather_data/weather_batch_{i}.csv.gz"
    s3_client.put_object(
        Bucket=s3_bucket,
        Key=s3_key,
        Body=gzip.compress(raw_text.encode())
    )

## Ingestion

In [8]:
from tenacity import retry, stop_after_attempt, wait_exponential
import requests
from tenacity import retry_if_exception_type

In [9]:
batch_size = 30

batches = [airport_list[i:i + batch_size] for i in range(0, len(airport_list), batch_size)]

In [10]:
@retry(
    stop=stop_after_attempt(3), 
    wait=wait_exponential(min=2, max=30),
    retry=retry_if_exception_type((requests.exceptions.RequestException, requests.exceptions.Timeout)),
    reraise=True
)
def fetch_and_write_batch(batch):
    data = fetch_batch(batch)
    write_to_datalake(raw_text=data, i=hash(tuple(batch)) % 10_000)
    return len(batch)

In [11]:
failed_batches = []
REQUEST_DELAY_SECONDS = 3

for batch in batches:
    try:
        fetch_and_write_batch(batch)
        print(f"Wrote batch {batch}")
    except Exception as e:
        print(f"Batch {batch} failed after retries: {e}")
        failed_batches.append(batch)
    time.sleep(REQUEST_DELAY_SECONDS)

Wrote batch ['BGM', 'PSE', 'INL', 'DLG', 'MSY', 'PPG', 'GEG', 'BUR', 'SNA', 'GRB', 'GTF', 'FOD', 'IDA', 'GRR', 'PVU', 'JLN', 'EUG', 'PSG', 'ATY', 'PVD', 'MYR', 'GSO', 'OAK', 'EAR', 'MSN', 'FAR', 'FSM', 'MQT', 'COD', 'BTM']
Wrote batch ['SCC', 'ESC', 'DCA', 'RFD', 'CID', 'MLU', 'LWS', 'SWO', 'GTR', 'PIB', 'WRG', 'HLN', 'CIU', 'IAG', 'DDC', 'LEX', 'RDM', 'JMS', 'ORF', 'SCE', 'KTN', 'EVV', 'GUF', 'CWA', 'CRW', 'VCT', 'SAV', 'GCK', 'CDV', 'TRI']
Wrote batch ['CMH', 'SPN', 'LBF', 'ADK', 'CAK', 'TYR', 'CHO', 'MOB', 'PNS', 'DIK', 'LIH', 'IAH', 'FMN', 'HNL', 'SLN', 'SHV', 'GST', 'ERI', 'SJC', 'CVG', 'TOL', 'LGA', 'BUF', 'TLH', 'CDC', 'ACT', 'HPN', 'RDD', 'AUS', 'MLI']
Wrote batch ['GCC', 'SJU', 'ATW', 'DHN', 'LGB', 'AVL', 'GJT', 'BFL', 'RIW', 'GFK', 'RNO', 'SRQ', 'EYW', 'SBN', 'BJI', 'TTN', 'JAC', 'RST', 'CHS', 'HGR', 'RSW', 'TUL', 'HRL', 'ISP', 'AMA', 'BOS', 'MAF', 'PQI', 'MLB', 'EWR']
Wrote batch ['LAS', 'BIS', 'JAN', 'ITO', 'FAI', 'IMT', 'XNA', 'HHH', 'DLH', 'HYA', 'DEN', 'RHI', 'EWN', 'SGU

In [14]:
len(failed_batches)

1

In [33]:
for fb in failed_batches:
    try:
        fetch_and_write_batch(fb)
        print(f"Wrote batch {fb}")
    except Exception as e:
        print(f"Batch {fb} failed after retries: {e}")
    time.sleep(REQUEST_DELAY_SECONDS)

Wrote batch ['KOA', 'EAU', 'GPT', 'MGM', 'JST', 'HOU', 'DTW', 'TYS', 'CHA', 'ADQ', 'YUM', 'ONT', 'MDT', 'FWA', 'JAX', 'STS', 'LAX', 'MSP', 'HTS', 'MOT', 'BGR', 'SIT', 'BTR', 'MCO', 'ROC', 'OTZ', 'SGF', 'AGS', 'WYS', 'SAN']
